# P1-E10.3 — Frozen CAUEEG Three-Endpoint External Validation on Dryad

## Goal

Apply the three fitted P1-E08. CAUEEG conventional benchmark models once to the corrected Dryad/P-ADIC participant cohort:

1. Normal vs AD
2. MCI vs AD
3. Normal–MCI–AD

No model is refitted, recalibrated, tuned or threshold-adjusted. Passing denotes protocol integrity rather than a required performance level.

> **P1-E10.3 PASSED — DRYAD FROZEN THREE-ENDPOINT EXTERNAL VALIDATION LOCKED**

## Context & Methods

### Prespecified rules

- Use the 179 empirical 19-channel HDF5 EEG objects locked by corrected P1-E10.1: 96 CN, 34 MCI and 49 AD.
- Use the documented Dryad source rate of 500 Hz and documented standard 19-channel order.
- Apply the frozen CAUEEG reference path: no second CAR, 1–40 Hz, resample to 200 Hz, 2-second edge trim.
- Generate non-overlapping 10-second epochs and apply the P1-E07.1 automatic QA thresholds.
- Retain 6–12 evenly distributed accepted epochs per participant.
- Keep diagnostic labels sealed until preprocessing and feature extraction are complete.
- Use the exact 646-feature P1-E08 schema and each artifact's frozen feature subset.
- Report endpoints separately; do not use the completed AHEPA outcomes to change this analysis.

### Key limitation

Dryad matrices do not carry BIDS channel labels. Channel order is inherited from the dataset documentation and locked before outcomes are evaluated. The `c1` class files are evaluated as the supplied resting-condition recordings; no outcome-guided interval selection is performed.

## Setup

### 1. Paths and frozen constants

In [ ]:
from pathlib import Path
from fractions import Fraction
import hashlib, json, platform, sys, warnings
import h5py, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, brier_score_loss, roc_curve)

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E10_1_ROOT=PAPER_ROOT/'p1_e10_1_dual_external_compatibility_protocol_lock'
E10_2_ROOT=PAPER_ROOT/'p1_e10_2_ahepa_frozen_normal_ad_external_validation'
E08_ROOT=PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
OUTPUT_ROOT=PAPER_ROOT/'p1_e10_3_dryad_frozen_three_endpoint_external_validation'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; ARRAYS=OUTPUT_ROOT/'epoch_arrays'
for p in (OUTPUT_ROOT,TABLES,FIGURES,ARRAYS): p.mkdir(parents=True,exist_ok=True)

E10_1_RUN=E10_1_ROOT/'run_record.json'; E10_2_RUN=E10_2_ROOT/'run_record.json'
E10_1_PROTOCOL=E10_1_ROOT/'p1_e10_1_LOCKED_EXTERNAL_VALIDATION_PROTOCOL.json'
DRYAD_MANIFEST=E10_1_ROOT/'tables'/'07_DRYAD_provisional_participant_manifest.csv'
ENDPOINT_TABLE=E10_1_ROOT/'tables'/'08_external_endpoint_availability.csv'
MODEL_PATHS={task:E08_ROOT/'models'/f'{task}_LOCKED_NONSPIKING_BENCHMARK.joblib'
             for task in ['normal_vs_ad','mci_vs_ad','three_class']}

SEED=20260828; SOURCE_FS=500.0; FS=200.0; BAND=(1.0,40.0); FILTER_ORDER=4
EDGE_TRIM_SECONDS=2.0; EPOCH_SECONDS=10.0; TARGET_EPOCHS=12; MIN_EPOCHS=6
PTP_LIMIT_UV=600.0; RMS_LIMIT_UV=120.0; EXTREME_UV=150.0
MAX_EXTREME_FRACTION=0.02; ROBUST_SCORE_LIMIT=5.0; FLAT_SD_UV=1e-6
MIN_RETENTION=0.90; BOOTSTRAP_REPS=2000
CHANNELS=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'low_gamma':(30,40)}
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],'three_class':['Normal','MCI','AD']}

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,'axes.titlesize':10,
 'axes.labelsize':9,'pdf.fonttype':42,'ps.fonttype':42}); sns.set_theme(style='whitegrid',context='paper')
def sha256_file(path,block_size=4*1024*1024):
    d=hashlib.sha256()
    with Path(path).open('rb') as h:
        for block in iter(lambda:h.read(block_size),b''): d.update(block)
    return d.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')
def robust_z(x):
    x=np.asarray(x,float); med=np.nanmedian(x); mad=np.nanmedian(np.abs(x-med))
    return np.zeros_like(x) if not np.isfinite(mad) or mad<1e-12 else (x-med)/(1.4826*mad)
print('Experiment: P1-E10.3'); print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate the frozen protocol, manifests and three models

In [ ]:
required=[E10_1_RUN,E10_2_RUN,E10_1_PROTOCOL,DRYAD_MANIFEST,ENDPOINT_TABLE,*MODEL_PATHS.values()]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Required locked inputs missing:\n'+'\n'.join(missing))
run10_1=json.loads(E10_1_RUN.read_text(encoding='utf-8')); run10_2=json.loads(E10_2_RUN.read_text(encoding='utf-8'))
protocol=json.loads(E10_1_PROTOCOL.read_text(encoding='utf-8'))
artifacts={task:joblib.load(path) for task,path in MODEL_PATHS.items()}
expected_hashes=run10_1.get('upstream_artifact_hashes',{})
checks=pd.DataFrame([
 {'check':'P1-E10.1 passed','pass':str(run10_1.get('status','')).startswith('P1-E10.1 PASSED'),'detail':run10_1.get('status','')},
 {'check':'P1-E10.2 completed before Dryad','pass':str(run10_2.get('status','')).startswith('P1-E10.2 PASSED'),'detail':'AHEPA result not used for decisions'},
 {'check':'Dryad declared READY','pass':run10_1.get('dataset_readiness',{}).get('DRYAD')=='READY','detail':run10_1.get('dataset_readiness',{})},
 {'check':'protocol hash authenticated','pass':sha256_file(E10_1_PROTOCOL)==run10_1.get('protocol_sha256'),'detail':sha256_file(E10_1_PROTOCOL)},
 {'check':'three model hashes authenticated','pass':all(sha256_file(p)==expected_hashes.get(f'model {t}') for t,p in MODEL_PATHS.items()),'detail':{t:sha256_file(p) for t,p in MODEL_PATHS.items()}},
 {'check':'artifact classes exact','pass':all(list(artifacts[t]['classes'])==TASKS[t] for t in TASKS),'detail':{t:list(artifacts[t]['classes']) for t in TASKS}},
])
save_table(checks,'01_input_authentication_checks'); display(checks)
if not checks['pass'].all(): raise AssertionError('STOP — FROZEN INPUT AUTHENTICATION FAILED')

### 3. Seal labels and create the diagnosis-blind Dryad processing manifest

In [ ]:
manifest=pd.read_csv(DRYAD_MANIFEST,dtype={'participant_id':str})
required_columns={'participant_id','path','source_hdf5_object','external_label'}
if not required_columns.issubset(manifest): raise KeyError(f'Dryad manifest missing {required_columns-set(manifest)}')
if manifest.participant_id.duplicated().any(): raise ValueError('Duplicate Dryad participant IDs')
label_lock=manifest[['participant_id','external_label']].copy()
label_lock['true_label']=label_lock.external_label.map({'CN':'Normal','MCI':'MCI','AD':'AD'})
if label_lock.true_label.isna().any(): raise ValueError('Unexpected labels in confirmatory Dryad manifest')
blind=manifest[['participant_id','path','source_hdf5_object']].copy()
save_table(blind,'02_DIAGNOSIS_BLIND_DRYAD_PROCESSING_MANIFEST'); save_table(label_lock,'03_SEALED_DRYAD_LABEL_LOCK')
display(label_lock.groupby('true_label').participant_id.nunique().rename('participants'))
print('Processing columns:',blind.columns.tolist())

## Results

### 4. Load each HDF5 signal and apply the frozen preprocessing/QA path

In [ ]:
def load_signal_uv(path,object_name):
    with h5py.File(path,'r') as h: data=np.asarray(h[object_name],dtype=float)
    if data.ndim!=2 or data.shape.count(19)!=1: raise ValueError(f'not_one_19_channel_axis_{data.shape}')
    if data.shape[0]!=19: data=data.T
    if data.shape[0]!=19: raise ValueError(f'orientation_failed_{data.shape}')
    centred=data-np.median(data,axis=1,keepdims=True); p95=float(np.percentile(np.abs(centred),95))
    if 1e-8<=p95<0.02: return data*1e6,'volts_to_microvolts',p95
    if 0.02<=p95<=10000: return data,'already_microvolts',p95
    raise ValueError(f'unresolved_amplitude_unit_p95_{p95}')

def preprocess(data_uv):
    x=signal.detrend(np.asarray(data_uv,float),axis=1,type='linear')
    sos=signal.butter(FILTER_ORDER,BAND,btype='bandpass',fs=SOURCE_FS,output='sos')
    x=signal.sosfiltfilt(sos,x,axis=1)
    ratio=Fraction(FS/SOURCE_FS).limit_denominator(1000)
    return signal.resample_poly(x,ratio.numerator,ratio.denominator,axis=1)
def make_epochs(x):
    trim=int(EDGE_TRIM_SECONDS*FS); length=int(EPOCH_SECONDS*FS); x=x[:,trim:x.shape[1]-trim]
    return np.asarray([x[:,i:i+length] for i in range(0,x.shape[1]-length+1,length)])
def qa_metrics(epochs):
    finite=np.isfinite(epochs).all(axis=(1,2)); sd=np.std(epochs,axis=2); flat=(sd<=FLAT_SD_UV).sum(axis=1)
    ptp=np.ptp(epochs,axis=2).max(axis=1); rms=np.sqrt(np.mean(epochs**2,axis=(1,2)))
    step=np.max(np.abs(np.diff(epochs,axis=2)),axis=(1,2)); extreme=np.mean(np.abs(epochs)>=EXTREME_UV,axis=(1,2))
    robust=np.maximum.reduce([np.abs(robust_z(np.log(ptp+1e-9))),np.abs(robust_z(np.log(rms+1e-9))),
      np.abs(robust_z(np.log(step+1e-9))),np.abs(robust_z(extreme+1e-9))])
    accepted=finite&(flat==0)&(ptp<PTP_LIMIT_UV)&(rms<RMS_LIMIT_UV)&(extreme<MAX_EXTREME_FRACTION)&(robust<ROBUST_SCORE_LIMIT)
    return pd.DataFrame({'finite':finite,'flat_channels':flat,'ptp_uv':ptp,'rms_uv':rms,'max_step_uv':step,
      'extreme_fraction':extreme,'robust_multimetric_score':robust,'accepted':accepted})
def evenly(indices,target):
    indices=np.asarray(indices,int)
    return indices if len(indices)<=target else indices[np.linspace(0,len(indices)-1,target).round().astype(int)]

participant_rows=[]; epoch_rows=[]
for number,row in enumerate(blind.itertuples(index=False),1):
    try:
        data_uv,unit_rule,raw_p95=load_signal_uv(row.path,row.source_hdf5_object)
        candidate=make_epochs(preprocess(data_uv)); qa=qa_metrics(candidate)
        accepted=np.flatnonzero(qa.accepted.to_numpy()); selected=evenly(accepted,TARGET_EPOCHS); selected_set=set(selected.tolist())
        for i,q in qa.iterrows(): epoch_rows.append({'participant_id':row.participant_id,'epoch_index':i,**q.to_dict(),'selected':i in selected_set})
        if len(selected)<MIN_EPOCHS: raise ValueError(f'only_{len(selected)}_accepted_epochs')
        kept=candidate[selected].astype('float32'); array_path=ARRAYS/f'{row.participant_id}_LOCKED_DRYAD_EPOCHS.npz'
        np.savez_compressed(array_path,epochs_uv=kept,fs_Hz=FS,channels=np.asarray(CHANNELS,dtype='U4'))
        participant_rows.append({'participant_id':row.participant_id,'unit_rule':unit_rule,'raw_p95':raw_p95,
          'source_samples':data_uv.shape[1],'candidate_epochs':len(candidate),'accepted_epochs':len(accepted),
          'selected_epochs':len(selected),'processing_success':True,'array_path':str(array_path),
          'array_sha256':sha256_file(array_path),'error':''})
    except Exception as exc:
        participant_rows.append({'participant_id':row.participant_id,'unit_rule':'FAILED','raw_p95':np.nan,
          'source_samples':0,'candidate_epochs':0,'accepted_epochs':0,'selected_epochs':0,
          'processing_success':False,'array_path':'','array_sha256':'','error':f'{type(exc).__name__}: {exc}'})
    if number%20==0 or number==len(blind): print(f'[{number:03d}/{len(blind)}] Dryad participants processed')
participant_audit=pd.DataFrame(participant_rows); epoch_audit=pd.DataFrame(epoch_rows)
save_table(participant_audit,'04_DRYAD_PARTICIPANT_PREPROCESSING_AUDIT'); save_table(epoch_audit,'05_DRYAD_AUTOMATED_EPOCH_QA_AUDIT')
display(participant_audit.processing_success.value_counts(dropna=False)); display(participant_audit.unit_rule.value_counts(dropna=False));
display(participant_audit.loc[~participant_audit.processing_success].head())

### 5. Extract and authenticate the exact P1-E08 feature schema

In [ ]:
def integrate_psd(psd,freqs,low,high):
    mask=(freqs>=low)&(freqs<high if high<40 else freqs<=high)
    return np.trapezoid(psd[...,mask],freqs[mask],axis=-1)
def epoch_features(epochs):
    freqs,psd=signal.welch(np.asarray(epochs,float),fs=FS,nperseg=int(4*FS),axis=-1)
    total=integrate_psd(psd,freqs,1,40)+1e-18; values=[]; names=[]
    for band,(low,high) in BANDS.items():
        power=integrate_psd(psd,freqs,low,high)
        for matrix,prefix in [(np.log10(power+1e-18),f'logbp_{band}'),(power/total,f'relbp_{band}')]:
            values.append(matrix); names += [f'{prefix}_{ch}' for ch in CHANNELS]
    use=(freqs>=1)&(freqs<=40); prob=psd[...,use]/(psd[...,use].sum(axis=-1,keepdims=True)+1e-18)
    entropy=-(prob*np.log(prob+1e-18)).sum(axis=-1)/np.log(prob.shape[-1]); cumulative=np.cumsum(psd[...,use],axis=-1)
    edge=(cumulative>=.95*cumulative[...,-1][...,None]).argmax(axis=-1); sef95=freqs[use][edge]
    alpha=(freqs>=8)&(freqs<=13); alpha_peak=freqs[alpha][np.argmax(psd[...,alpha],axis=-1)]
    for matrix,prefix in [(entropy,'spectral_entropy'),(sef95,'sef95_Hz'),(alpha_peak,'alpha_peak_Hz')]:
        values.append(matrix); names += [f'{prefix}_{ch}' for ch in CHANNELS]
    dx=np.diff(epochs,axis=-1); ddx=np.diff(dx,axis=-1); activity=np.var(epochs,axis=-1)
    mobility=np.sqrt(np.var(dx,axis=-1)/(activity+1e-18)); mobility_dx=np.sqrt(np.var(ddx,axis=-1)/(np.var(dx,axis=-1)+1e-18))
    complexity=mobility_dx/(mobility+1e-18); line_length=np.mean(np.abs(dx),axis=-1)
    for matrix,prefix in [(np.log10(activity+1e-18),'hjorth_log_activity'),(mobility,'hjorth_mobility'),
                          (complexity,'hjorth_complexity'),(line_length,'line_length')]:
        values.append(matrix); names += [f'{prefix}_{ch}' for ch in CHANNELS]
    return np.concatenate(values,axis=1),names

rows=[]; feature_audit=[]; aggregate_names=None
for row in participant_audit.loc[participant_audit.processing_success].itertuples(index=False):
    try:
        if sha256_file(row.array_path)!=row.array_sha256: raise ValueError('array_hash_mismatch')
        with np.load(row.array_path,allow_pickle=False) as p: epochs=np.asarray(p['epochs_uv'],float)
        matrix,names=epoch_features(epochs); current=[f'median__{x}' for x in names]+[f'iqr__{x}' for x in names]
        if aggregate_names is None: aggregate_names=current
        elif current!=aggregate_names: raise ValueError('feature_schema_changed')
        vector=np.concatenate([np.median(matrix,axis=0),np.quantile(matrix,.75,axis=0)-np.quantile(matrix,.25,axis=0)])
        rows.append({'participant_id':row.participant_id,'vector':vector}); feature_audit.append({'participant_id':row.participant_id,'success':True,'error':''})
    except Exception as exc: feature_audit.append({'participant_id':row.participant_id,'success':False,'error':repr(exc)})
feature_audit=pd.DataFrame(feature_audit); save_table(feature_audit,'06_DRYAD_FEATURE_EXTRACTION_AUDIT')
if not feature_audit.success.all(): raise RuntimeError('Feature extraction failed; do not evaluate.')
ids=[r['participant_id'] for r in rows]; X=np.vstack([r['vector'] for r in rows]).astype('float32')
schema_by_task={t:np.array_equal(np.asarray(aggregate_names)[np.asarray(a['feature_indices'],int)],np.asarray(a['feature_names']).astype(str)) for t,a in artifacts.items()}
if not all(schema_by_task.values()): raise AssertionError(f'Feature schema mismatch: {schema_by_task}')
if not np.isfinite(X).all(): raise ValueError('Nonfinite features')
np.savez_compressed(OUTPUT_ROOT/'DRYAD_LOCKED_NONSPIKING_FEATURE_MATRIX.npz',X=X,participant_id=np.asarray(ids,dtype='U20'),feature_names=np.asarray(aggregate_names,dtype='U96'))
print('Feature matrix:',X.shape,'schemas:',schema_by_task)

### 6. Perform the three single-shot frozen endpoint evaluations

In [ ]:
base=pd.DataFrame({'participant_id':ids}).merge(label_lock[['participant_id','true_label']],on='participant_id',validate='one_to_one')
prediction_rows=[]; metric_rows=[]; ci_rows=[]; report_rows=[]
for task,classes_task in TASKS.items():
    use=base.true_label.isin(classes_task).to_numpy(); frame=base.loc[use].copy(); artifact=artifacts[task]
    columns=np.asarray(artifact['feature_indices'],int); estimator=artifact['estimator']; selected=X[use][:,columns]
    pred=estimator.predict(selected); frame['predicted_label']=pred; frame['correct']=frame.true_label.eq(frame.predicted_label); frame['task']=task
    proba=None
    if hasattr(estimator,'predict_proba'):
        proba=estimator.predict_proba(selected); model_classes=np.asarray(estimator.classes_).astype(str)
        for j,label in enumerate(model_classes): frame[f'probability_{label}']=proba[:,j]
    prediction_rows.extend(frame.to_dict('records'))
    yt=frame.true_label.to_numpy(); yp=np.asarray(pred)
    observed={'accuracy':accuracy_score(yt,yp),'balanced_accuracy':balanced_accuracy_score(yt,yp),
              'macro_f1':f1_score(yt,yp,average='macro'),'mcc':matthews_corrcoef(yt,yp)}
    if len(classes_task)==2:
        neg=classes_task[0]; yb=(yt=='AD').astype(int); pi=proba[:,int(np.flatnonzero(model_classes=='AD')[0])]
        tn,fp,fn,tp=confusion_matrix(yt,yp,labels=[neg,'AD']).ravel()
        observed.update({'sensitivity_AD':tp/(tp+fn),'specificity_'+neg:tn/(tn+fp),
                         'roc_auc':roc_auc_score(yb,pi),'average_precision':average_precision_score(yb,pi),
                         'brier_score':brier_score_loss(yb,pi)})
    elif proba is not None:
        # sklearn probability columns already follow estimator.classes_; preserve that exact order.
        observed['macro_ovr_roc_auc']=roc_auc_score(yt,proba,labels=model_classes,multi_class='ovr',average='macro')
    rng=np.random.default_rng(SEED); strata={c:np.flatnonzero(yt==c) for c in classes_task}; boots=[]
    for _ in range(BOOTSTRAP_REPS):
        idx=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes_task]); bt=yt[idx]; bp=yp[idx]
        row={'accuracy':accuracy_score(bt,bp),'balanced_accuracy':balanced_accuracy_score(bt,bp),'macro_f1':f1_score(bt,bp,average='macro'),'mcc':matthews_corrcoef(bt,bp)}
        if len(classes_task)==2:
            bpi=pi[idx]; bbin=(bt=='AD').astype(int); tn,fp,fn,tp=confusion_matrix(bt,bp,labels=[classes_task[0],'AD']).ravel()
            row.update({'sensitivity_AD':tp/(tp+fn),'specificity_'+classes_task[0]:tn/(tn+fp),
                        'roc_auc':roc_auc_score(bbin,bpi),'average_precision':average_precision_score(bbin,bpi),'brier_score':brier_score_loss(bbin,bpi)})
        elif proba is not None:
            row['macro_ovr_roc_auc']=roc_auc_score(bt,proba[idx],labels=model_classes,multi_class='ovr',average='macro')
        boots.append(row)
    boots=pd.DataFrame(boots)
    for name,value in observed.items(): ci_rows.append({'task':task,'metric':name,'estimate':value,'ci95_low':boots[name].quantile(.025),'ci95_high':boots[name].quantile(.975)})
    metric_rows.append({'task':task,'participants':len(frame),**observed})
    rep=classification_report(yt,yp,labels=classes_task,output_dict=True,zero_division=0)
    for label,values in rep.items():
        if isinstance(values,dict): report_rows.append({'task':task,'label':label,**values})

predictions=pd.DataFrame(prediction_rows); metrics=pd.DataFrame(metric_rows); intervals=pd.DataFrame(ci_rows); reports=pd.DataFrame(report_rows)
save_table(predictions,'07_LOCKED_DRYAD_PARTICIPANT_PREDICTIONS'); save_table(metrics,'08_DRYAD_EXTERNAL_VALIDATION_METRICS')
save_table(intervals,'09_DRYAD_PARTICIPANT_BOOTSTRAP_INTERVALS'); save_table(reports,'10_DRYAD_CLASSIFICATION_REPORTS')
display(metrics); display(intervals)

### 7. Publication-ready Dryad external-validation figures

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.7))
for ax,(task,classes_task) in zip(axes,TASKS.items()):
    use=predictions.loc[predictions.task.eq(task)]; matrix=confusion_matrix(use.true_label,use.predicted_label,labels=classes_task)
    sns.heatmap(matrix,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,xticklabels=classes_task,yticklabels=classes_task)
    ax.set(title=task.replace('_',' '),xlabel='Predicted',ylabel='True')
savefig(fig,'Figure_01_DRYAD_external_confusion_matrices',12,3.7); plt.show()

plot=metrics.melt(id_vars=['task','participants'],value_vars=['balanced_accuracy','macro_f1','mcc'],var_name='metric',value_name='score')
fig,ax=plt.subplots(); sns.barplot(data=plot,x='task',y='score',hue='metric',ax=ax)
ax.set(ylim=(-.1,1),xlabel='',ylabel='Participant-level score',title='Frozen CAUEEG models on Dryad'); ax.tick_params(axis='x',rotation=15); ax.legend(frameon=False)
savefig(fig,'Figure_02_DRYAD_external_performance',8,4); plt.show()

## Checks

### 8. Lock the Dryad results and provenance

In [ ]:
retention=participant_audit.processing_success.mean(); unit_modes=participant_audit.loc[participant_audit.processing_success,'unit_rule'].unique()
final_checks=pd.DataFrame([
 {'check':'all upstream gates passed','pass':checks['pass'].all(),'detail':'locked protocol and artifacts'},
 {'check':'preprocessing diagnosis-blind','pass':set(blind.columns)=={'participant_id','path','source_hdf5_object'},'detail':blind.columns.tolist()},
 {'check':'corrected 179-object manifest used','pass':len(blind)==179,'detail':len(blind)},
 {'check':'minimum participant retention reached','pass':retention>=MIN_RETENTION,'detail':retention},
 {'check':'amplitude-unit rule consistent','pass':len(unit_modes)==1,'detail':unit_modes.tolist()},
 {'check':'minimum accepted epochs met','pass':participant_audit.loc[participant_audit.processing_success].selected_epochs.ge(MIN_EPOCHS).all(),'detail':participant_audit.loc[participant_audit.processing_success].selected_epochs.min()},
 {'check':'all three feature schemas exact','pass':all(schema_by_task.values()),'detail':schema_by_task},
 {'check':'all three endpoints evaluated','pass':set(metrics.task)==set(TASKS),'detail':metrics.task.tolist()},
 {'check':'one prediction per participant/task','pass':not predictions.duplicated(['task','participant_id']).any(),'detail':len(predictions)},
 {'check':'no model fitting performed','pass':True,'detail':'predict and predict_proba only'},
 {'check':'no tuning/recalibration/threshold change','pass':True,'detail':'frozen estimator rules'},
 {'check':'documented source channel order locked','pass':CHANNELS==['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2'],'detail':'19-channel Dryad order'},
])
save_table(final_checks,'11_final_integrity_checks'); display(final_checks)
status=('P1-E10.3 PASSED — DRYAD FROZEN THREE-ENDPOINT EXTERNAL VALIDATION LOCKED' if final_checks['pass'].all()
        else 'STOP — P1-E10.3 CHECKS FAILED; DO NOT INTERPRET DRYAD PERFORMANCE')
pred_path=TABLES/'07_LOCKED_DRYAD_PARTICIPANT_PREDICTIONS.csv'; metric_path=TABLES/'08_DRYAD_EXTERNAL_VALIDATION_METRICS.csv'
run_record={'experiment':'P1-E10.3 frozen CAUEEG three-endpoint external validation on Dryad','timestamp_utc':pd.Timestamp.utcnow().isoformat(),
 'status':status,'source_p1_e10_1_run_sha256':sha256_file(E10_1_RUN),'source_protocol_sha256':sha256_file(E10_1_PROTOCOL),
 'source_model_hashes':{t:sha256_file(p) for t,p in MODEL_PATHS.items()},'input_participants':len(blind),
 'evaluated_participants':int(participant_audit.processing_success.sum()),'retention_fraction':retention,
 'model_refit':False,'threshold_changed':False,'calibration_changed':False,
 'predictions_csv':str(pred_path),'predictions_sha256':sha256_file(pred_path),'metrics_csv':str(metric_path),
 'metrics_sha256':sha256_file(metric_path),'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not final_checks['pass'].all(): raise AssertionError(status)